In [1]:
import os

# List files in current directory
print(os.listdir())

# If you know the filename, replace below:
file_path = "/content/C15_1111.mp3"
if os.path.exists(file_path):
    print("File exists ✅")
    print("File size (MB):", os.path.getsize(file_path) / 1024 / 1024)
else:
    print("File not found ❌")

['.config', 'C15_1111.mp3', 'sample_data']
File exists ✅
File size (MB): 4.0


In [1]:
import torch
import whisperx

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("WhisperX OK")

Torch: 2.8.0+cu128
CUDA: True
WhisperX OK


In [ ]:
from huggingface_hub import login
login("")

In [1]:
# # ----------------------------
# # 1️⃣ Install compatible packages
# # ----------------------------
# !pip uninstall -y torchvision torchaudio -q
# !pip install -q torch==2.8.0+cu128 torchvision==0.23.0+cu128 torchaudio==2.8.0+cu128 --index-url https://download.pytorch.org/whl/cu128
# !pip install -q transformers==4.48.0
# !pip install -q whisperx==3.8.1
# !pip install -q pyannote.audio==4.10.0  # Required for diarization pipeline


ERROR: Could not find a version that satisfies the requirement pyannote.audio==4.10.0 (from versions: 0.0.1, 1.1, 1.1.1, 1.1.2, 2.0.1, 2.1.1, 3.0.0, 3.0.1, 3.1.0, 3.1.1, 3.2.0, 3.3.0, 3.3.1, 3.3.2, 3.4.0, 4.0.0, 4.0.1, 4.0.2, 4.0.3, 4.0.4)
ERROR: No matching distribution found for pyannote.audio==4.10.0


In [ ]:

# ----------------------------
# 2️⃣ Imports
# ----------------------------
import whisperx
import torch
from whisperx.diarize import DiarizationPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
audio_file = "C15_1111.mp3"

# ----------------------------
# 3️⃣ Load WhisperX model
# ----------------------------
print("Loading WhisperX model...")
model = whisperx.load_model("large-v2", device)

# ----------------------------
# 4️⃣ Transcribe audio
# ----------------------------
print("Transcribing audio...")
result = model.transcribe(audio_file)

# ----------------------------
# 5️⃣ Word-level alignment
# ----------------------------
print("Aligning words...")
model_a, metadata = whisperx.load_align_model(
    language_code=result["language"],
    device=device
)
result = whisperx.align(
    result["segments"],
    model_a,
    metadata,
    audio_file,
    device
)

# ----------------------------
# 6️⃣ Speaker diarization (gated model)
# ----------------------------
# Use your Hugging Face token for access to pyannote/speaker-diarization-community-1
HF_TOKEN = ""  # <-- replace with your token

print("Running speaker diarization...")
diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)
diarize_segments = diarize_model(
    audio_file,
    min_speakers=2,
    max_speakers=2
)

# ----------------------------
# 7️⃣ Assign speakers to words
# ----------------------------
result = whisperx.assign_word_speakers(diarize_segments, result)

# ----------------------------
# Print transcript with speaker labels
# ----------------------------
for segment in result["segments"]:
    # Assign speaker if available, otherwise default
    speaker = segment.get("speaker", "UNKNOWN")
    print(f"[{segment['start']:.2f}-{segment['end']:.2f}] {speaker}: {segment['text']}")

# ----------------------------
# word-level detail
# ----------------------------
print("\nWord-level transcript with speakers:")
for word in result["words"]:
    speaker = word.get("speaker", "UNKNOWN")
    print(f"[{word['start']:.2f}-{word['end']:.2f}] {speaker}: {word['word']}")


/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.8.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
          

Loading WhisperX model...


/usr/local/lib/python3.12/dist-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoi

2026-02-25 15:27:58 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-02-25 15:27:58 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


...

/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(


2026-02-25 15:28:01 - whisperx.asr - INFO - Detected language: en (0.86) in first 30s of audio
Aligning words...
Running speaker diarization...
2026-02-25 15:28:38 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


[1.26-2.57] SPEAKER_00:  Good evening, ma'am.
[2.59-3.19] SPEAKER_00: Good evening.
[3.31-4.91] SPEAKER_00: OK, I'll just share my screen.
[5.57-9.66] SPEAKER_00: Hopefully, it will not have a problem right now.
[9.68-10.16] SPEAKER_00: OK.
[11.17-12.45] SPEAKER_00: So tell me, can you see the screen?
[15.36-17.28] SPEAKER_00: Yes, ma'am.
[17.64-17.70] UNKNOWN: OK.
[18.26-20.44] SPEAKER_00:  So now let's play a game together.
[21.24-29.33] SPEAKER_00: I will be this woman who has a red computer and you will be this boy who has a blue computer.
[29.35-32.51] SPEAKER_00: You will choose a story that will show on your blue computer.
[32.91-35.96] SPEAKER_00: However, I will not see it on my red computer.
[35.98-39.84] SPEAKER_00: Then you look closely at the story and try to tell it yourself, okay?
[39.86-43.28] SPEAKER_00: I'm looking forward to it and I'm curious about the story you're going to tell.
[45.10-47.91] SPEAKER_00:  So look, there are three different envelopes here.
[48.33-50

KeyError: 'words'

In [1]:
# ----------------------------
# Install required packages (specific versions for compatibility)
# ----------------------------
!pip uninstall -y torchvision torchaudio -q
!pip install -q torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q transformers==4.48.0
!pip install -q whisperx==3.8.1
!pip install -q pyannote.audio==4.0.4
!pip install -q torchcodec==0.6.4  # optional if you want TorchCodec fixes

ERROR: Could not find a version that satisfies the requirement torchcodec==0.6.4 (from versions: 0.0.0.dev0, 0.0.1, 0.0.2, 0.0.3, 0.1.0, 0.1.1, 0.2.0, 0.2.1, 0.3.0, 0.4.0, 0.5, 0.6.0, 0.7.0, 0.8.0, 0.8.1, 0.9.0, 0.9.1, 0.10.0)
ERROR: No matching distribution found for torchcodec==0.6.4


In [ ]:
# ----------------------------
# Imports
# ----------------------------
import torch
import torchaudio
import whisperx
from whisperx.diarize import DiarizationPipeline
import numpy as np

# ----------------------------
# Config
# ----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
audio_file = "/content/C03_1003.mp3"
hf_token = ""

# ----------------------------
# Load WhisperX model
# ----------------------------
model = whisperx.load_model("large-v3", device)

# ----------------------------
# Transcribe (force English)
# ----------------------------
result = model.transcribe(audio_file, language="en")  # explicitly set language

# ----------------------------
# Load alignment model
# ----------------------------
model_a, metadata = whisperx.load_align_model(
    language_code="en",
    device=device
)

# ----------------------------
# Align words
# ----------------------------
result = whisperx.align(
    result["segments"],
    model_a,
    metadata,
    audio_file,
    device
)


#
# ----------------------------
# Preload audio and convert to mono numpy array
# ----------------------------
waveform, sample_rate = torchaudio.load(audio_file)  # shape: [channels, samples]
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)  # convert to mono
audio_np = waveform.squeeze(0).numpy()  # 1D NumPy array

# ----------------------------
# Run speaker diarization (force 2 speakers)
# ----------------------------
diarize_model = DiarizationPipeline(token=hf_token, device=device)
diarize_segments = diarize_model(
    audio_np,       # 1D numpy array only
    min_speakers=2,
    max_speakers=2

)

# ----------------------------
# Assign speakers to words
# ----------------------------
result = whisperx.assign_word_speakers(diarize_segments, result)

# ----------------------------
# Print segment-level transcript safely
# ----------------------------
print("Segment-level transcript with speakers:")
for segment in result.get("segments", []):
    speaker = segment.get("speaker", "UNKNOWN")
    text = segment.get("text", "")
    start = segment.get("start", 0.0)
    end = segment.get("end", 0.0)
    print(f"[{start:.2f}-{end:.2f}] {speaker}: {text}")

# ----------------------------
# Print word-level transcript if available
# ----------------------------
if "words" in result:
    print("\nWord-level transcript with speakers:")
    for word in result["words"]:
        speaker = word.get("speaker", "UNKNOWN")
        start = word.get("start", 0.0)
        end = word.get("end", 0.0)
        text = word.get("word", "")
        print(f"[{start:.2f}-{end:.2f}] {speaker}: {text}")
else:
    print("\nWord-level transcript not available for this audio/model.")

/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.8.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
          

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

2026-02-25 16:07:16 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-02-25 16:07:16 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issue

2026-02-25 16:08:18 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


Segment-level transcript with speakers:
[0.96-2.24] SPEAKER_00:  It is good.
[2.90-3.52] SPEAKER_00: It is good?
[3.60-3.96] SPEAKER_00: Okay.
[4.04-6.65] SPEAKER_00: I heard you are having your new classes now.
[6.71-11.53] SPEAKER_00: How does it feel like to be in, you know, which class are you in right now?
[11.57-12.09] SPEAKER_00: Five, right?
[14.18-15.02] SPEAKER_00: Yes, ma'am, five.
[15.08-15.86] SPEAKER_00: Yeah, yeah.
[16.84-20.14] SPEAKER_00: So how does it feel like growing up, going to class five?
[20.16-21.47] SPEAKER_00: Is it more pressurizing?
[23.77-24.31] SPEAKER_00: No, ma'am.
[25.23-26.83] UNKNOWN: Oh, okay.
[26.85-29.08] SPEAKER_00: So I'll introduce myself.
[29.78-32.33] SPEAKER_00:  Oh, nice, nice, nice.
[32.35-33.63] SPEAKER_00: Okay, so I'll introduce myself.
[34.21-35.22] SPEAKER_00: I'm Firdaush Shireen.
[35.86-41.65] SPEAKER_00: I have also been a student of MPS since I was in class five and then finished my 12th also there.
[42.13-45.58] SPEAKER_00: So I

In [2]:
# ----------------------------
# Save transcript to a TXT file
# ----------------------------

output_file = "transcription1003.txt"  # You can change the filename

with open(output_file, "w", encoding="utf-8") as f:
    for segment in result.get("segments", []):
        speaker = segment.get("speaker", "UNKNOWN")
        text = segment.get("text", "")
        start = segment.get("start", 0.0)
        end = segment.get("end", 0.0)
        f.write(f"[{start:.2f}-{end:.2f}] {speaker}: {text}\n")

print(f"Transcript saved to {output_file}")

Transcript saved to transcription1003.txt
